In [ ]:
# ============================================================
# CELLULE 1 : Installation des dépendances
# ============================================================

!pip install -q langchain langchain-community qdrant-client \
    sentence-transformers lxml tqdm

print("✅ Dépendances installées !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ Dépendances installées !


In [ ]:
# Cellule 1 — Monter le Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ============================================================
# CELLULE 2 : Parsing du XML DrugBank
# ============================================================

import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_drugbank_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = {"db": "http://www.drugbank.ca"}

    # Fonction helper corrigée — vérifie element ET son texte
    def safe_text(el):
        return el.text.strip() if el is not None and el.text else ""

    drugs = []

    for drug in tqdm(root.findall("db:drug", ns), desc="Parsing DrugBank"):

        def get(tag):
            el = drug.find(f"db:{tag}", ns)
            return safe_text(el)

        name  = get("name")
        db_id = ""
        for did in drug.findall("db:drugbank-id", ns):
            if did.attrib.get("primary") == "true":
                db_id = did.text.strip() if did.text else ""
                break

        indication  = get("indication")
        pharmacodyn = get("pharmacodynamics")
        mechanism   = get("mechanism-of-action")
        toxicity    = get("toxicity")
        half_life   = get("half-life")
        absorption  = get("absorption")

        synonyms = [
            s.text.strip()
            for s in drug.findall("db:synonyms/db:synonym", ns)
            if s.text
        ]

        brands = [
            safe_text(p.find("db:name", ns))
            for p in drug.findall("db:products/db:product", ns)
            if safe_text(p.find("db:name", ns))
        ]

        # ✅ Correction ici — safe_text sur chaque sous-élément
        dosages = []
        for d in drug.findall("db:dosages/db:dosage", ns):
            form     = safe_text(d.find("db:form", ns))
            route    = safe_text(d.find("db:route", ns))
            strength = safe_text(d.find("db:strength", ns))
            dosages.append({
                "form":     form,
                "route":    route,
                "strength": strength
            })

        categories = [
            safe_text(c.find("db:category", ns))
            for c in drug.findall("db:categories/db:category", ns)
            if safe_text(c.find("db:category", ns))
        ]

        if not name:
            continue

        drugs.append({
            "drugbank_id":  db_id,
            "name":         name,
            "synonyms":     synonyms,
            "brands":       brands,
            "indication":   indication,
            "pharmacodyn":  pharmacodyn,
            "mechanism":    mechanism,
            "toxicity":     toxicity,
            "half_life":    half_life,
            "absorption":   absorption,
            "dosages":      dosages,
            "categories":   categories,
        })

    print(f"\n✅ {len(drugs)} médicaments parsés avec succès !")
    return drugs

# --- Lancement du parsing ---
XML_PATH = "/content/drive/MyDrive/OCR_Prescription/drugbank.xml"
drugs = parse_drugbank_xml(XML_PATH)

# --- Aperçu du premier médicament ---
import json
print(json.dumps(drugs[0], indent=2, ensure_ascii=False))

Parsing DrugBank: 100%|██████████| 10562/10562 [00:12<00:00, 866.33it/s]



✅ 10562 médicaments parsés avec succès !
{
  "drugbank_id": "DB00001",
  "name": "Lepirudin",
  "synonyms": [
    "Hirudin variant-1",
    "Lepirudin recombinant"
  ],
  "brands": [
    "Refludan"
  ],
  "indication": "For the treatment of heparin-induced thrombocytopenia",
  "pharmacodyn": "Lepirudin is used to break up clots and to reduce thrombocytopenia. It binds to thrombin and prevents thrombus or clot formation. It is a highly potent, selective, and essentially irreversible inhibitor of thrombin and clot-bond thrombin. Lepirudin requires no cofactor for its anticoagulant action. Lepirudin is a recombinant form of hirudin, an endogenous anticoagulant found in medicinal leeches.",
  "mechanism": "Lepirudin forms a stable non-covalent complex with alpha-thrombin, thereby abolishing its ability to cleave fibrinogen and initiate the clotting cascade. The inhibition of thrombin prevents the blood clotting cascade.",
  "toxicity": "In case of overdose (eg, suggested by excessively hig

In [ ]:
# ============================================================
# CELLULE 3 : Construction des documents texte pour le RAG
# ============================================================

def build_document(drug):
    """
    Transforme un dict médicament en texte structuré
    optimisé pour l'embedding et la recherche sémantique.
    """
    lines = []

    lines.append(f"Médicament : {drug['name']}")
    lines.append(f"ID DrugBank : {drug['drugbank_id']}")

    if drug["synonyms"]:
        lines.append(f"Synonymes : {', '.join(drug['synonyms'])}")

    if drug["brands"]:
        lines.append(f"Noms commerciaux : {', '.join(drug['brands'])}")

    if drug["indication"]:
        lines.append(f"Indication : {drug['indication']}")

    if drug["mechanism"]:
        lines.append(f"Mécanisme d'action : {drug['mechanism']}")

    if drug["pharmacodyn"]:
        lines.append(f"Pharmacodynamique : {drug['pharmacodyn']}")

    if drug["toxicity"]:
        lines.append(f"Toxicité / Surdosage : {drug['toxicity']}")

    if drug["half_life"]:
        lines.append(f"Demi-vie : {drug['half_life']}")

    if drug["dosages"]:
        dosage_strs = [
            f"{d['strength']} ({d['form']}, voie {d['route']})"
            for d in drug["dosages"] if d["strength"]
        ]
        if dosage_strs:
            lines.append(f"Dosages disponibles : {' | '.join(dosage_strs)}")

    if drug["categories"]:
        lines.append(f"Catégories : {', '.join(drug['categories'][:5])}")

    return "\n".join(lines)


# --- Construction de tous les documents ---
documents = []
for drug in drugs:
    doc_text = build_document(drug)
    documents.append({
        "id":       drug["drugbank_id"],
        "name":     drug["name"],
        "text":     doc_text,
        "metadata": {
            "drugbank_id": drug["drugbank_id"],
            "name":        drug["name"],
            "synonyms":    drug["synonyms"],
            "brands":      drug["brands"],
            "categories":  drug["categories"],
            "dosages":     drug["dosages"],
        }
    })

print(f"✅ {len(documents)} documents construits !")
print("\n--- Exemple de document ---")
print(documents[0]["text"])

✅ 10562 documents construits !

--- Exemple de document ---
Médicament : Lepirudin
ID DrugBank : DB00001
Synonymes : Hirudin variant-1, Lepirudin recombinant
Noms commerciaux : Refludan
Indication : For the treatment of heparin-induced thrombocytopenia
Mécanisme d'action : Lepirudin forms a stable non-covalent complex with alpha-thrombin, thereby abolishing its ability to cleave fibrinogen and initiate the clotting cascade. The inhibition of thrombin prevents the blood clotting cascade.
Pharmacodynamique : Lepirudin is used to break up clots and to reduce thrombocytopenia. It binds to thrombin and prevents thrombus or clot formation. It is a highly potent, selective, and essentially irreversible inhibitor of thrombin and clot-bond thrombin. Lepirudin requires no cofactor for its anticoagulant action. Lepirudin is a recombinant form of hirudin, an endogenous anticoagulant found in medicinal leeches.
Toxicité / Surdosage : In case of overdose (eg, suggested by excessively high aPTT value

In [ ]:
# ============================================================
# CELLULE 4 : Embedding des documents
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

# Modèle d'embedding médical — meilleur que all-MiniLM pour le domaine biomédical
EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"

print(f"Chargement du modèle d'embedding : {EMBEDDING_MODEL} ...")
embedder = SentenceTransformer(EMBEDDING_MODEL)

# Extraction des textes
texts = [doc["text"] for doc in documents]

print(f"Calcul des embeddings pour {len(texts)} documents...")
print("(Cela peut prendre quelques minutes sur Colab CPU...)")

# Batch size 32 pour éviter les OOM
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings calculés ! Shape : {embeddings.shape}")
# Exemple : (11000, 768) → 11000 médicaments, vecteurs de 768 dimensions

Chargement du modèle d'embedding : pritamdeka/S-PubMedBert-MS-MARCO ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (9 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Calcul des embeddings pour 10562 documents...
(Cela peut prendre quelques minutes sur Colab CPU...)


Batches:   0%|          | 0/331 [00:00<?, ?it/s]


✅ Embeddings calculés ! Shape : (10562, 768)


In [ ]:
# ============================================================
# CELLULE 5 : Indexation dans Qdrant (en mémoire pour Colab)
# ============================================================

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(":memory:")
COLLECTION_NAME = "drugbank_medical"
VECTOR_SIZE     = embeddings.shape[1]

# ✅ Correction 1 — plus de recreate_collection
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

# Indexation par batches
BATCH_SIZE = 256
points = []

for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "drugbank_id": doc["metadata"]["drugbank_id"],
                "name":        doc["metadata"]["name"],
                "text":        doc["text"],
                "synonyms":    doc["metadata"]["synonyms"],
                "brands":      doc["metadata"]["brands"],
                "categories":  doc["metadata"]["categories"],
                "dosages":     doc["metadata"]["dosages"],
            }
        )
    )
    if len(points) == BATCH_SIZE or i == len(documents) - 1:
        client.upsert(collection_name=COLLECTION_NAME, points=points)
        points = []

# ✅ Correction 2 — points_count au lieu de vectors_count
info = client.get_collection(COLLECTION_NAME)
print(f"✅ {len(documents)} médicaments indexés dans Qdrant !")
print(f"Points indexés : {info.points_count}")

✅ 10562 médicaments indexés dans Qdrant !
Points indexés : 10562


In [ ]:
# ============================================================
# CELLULE 6 : Fonction de recherche RAG
# ============================================================

def search_drug(query: str, top_k: int = 3, score_threshold: float = 0.5):
    """
    Recherche hybride :
    1. Recherche exacte sur noms commerciaux et synonymes
    2. Recherche sémantique si rien trouvé
    """

    query_lower = query.lower().strip()

    # ── Étape 1 : recherche exacte sur brands/synonymes ──────────
    exact_match = None
    for doc in documents:
        brands   = [b.lower() for b in doc["metadata"]["brands"]]
        synonyms = [s.lower() for s in doc["metadata"]["synonyms"]]
        name     = doc["metadata"]["name"].lower()

        if query_lower in brands or query_lower in synonyms or query_lower == name:
            exact_match = {
                "name":        doc["metadata"]["name"],
                "drugbank_id": doc["metadata"]["drugbank_id"],
                "score":       1.0,  # Score parfait car match exact
                "dosages":     doc["metadata"]["dosages"],
                "categories":  doc["metadata"]["categories"],
                "synonyms":    doc["metadata"]["synonyms"],
                "brands":      doc["metadata"]["brands"],
                "text":        doc["text"],
                "match_type":  "exact"
            }
            break

    if exact_match:
        return [exact_match]

    # ── Étape 2 : recherche sémantique si pas de match exact ─────
    query_vector = embedder.encode(query).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        score_threshold=score_threshold
    ).points

    if not results:
        return []

    return [
        {
            "name":        r.payload["name"],
            "drugbank_id": r.payload["drugbank_id"],
            "score":       round(r.score, 4),
            "dosages":     r.payload["dosages"],
            "categories":  r.payload["categories"],
            "synonyms":    r.payload["synonyms"],
            "brands":      r.payload["brands"],
            "text":        r.payload["text"],
            "match_type":  "semantic"
        }
        for r in results
    ]


# ── Tests mis à jour ─────────────────────────────────────────────
test_queries = [
    "Lepirudin",
    "lepirudine",
    "lpirdn",
    "anticoagulant heparin induced thrombocytopenia",
    "Refludan",       # ← doit maintenant trouver Lepirudin via exact match
    "hirudin recombinant",
]

print("="*60)
print("TESTS DE RECHERCHE RAG — VERSION HYBRIDE")
print("="*60)

for query in test_queries:
    results = search_drug(query, top_k=1)
    if results:
        r = results[0]
        match_type = r.get("match_type", "semantic")
        icon = "✅" if r["score"] > 0.7 else "⚠️"
        print(f"\nQuery      : '{query}'")
        print(f"Trouvé     : {r['name']} (ID: {r['drugbank_id']})")
        print(f"Score      : {r['score']} {icon}")
        print(f"Match type : {match_type}")
    else:
        print(f"\nQuery      : '{query}'")
        print(f"Résultat   : ❌ Aucun médicament trouvé")

TESTS DE RECHERCHE RAG — VERSION HYBRIDE

Query      : 'Lepirudin'
Trouvé     : Lepirudin (ID: DB00001)
Score      : 1.0 ✅
Match type : exact

Query      : 'lepirudine'
Trouvé     : Lepirudin (ID: DB00001)
Score      : 0.9108 ✅
Match type : semantic

Query      : 'lpirdn'
Trouvé     : Rilapladib (ID: DB05119)
Score      : 0.8925 ✅
Match type : semantic

Query      : 'anticoagulant heparin induced thrombocytopenia'
Trouvé     : Argatroban (ID: DB00278)
Score      : 0.9192 ✅
Match type : semantic

Query      : 'Refludan'
Trouvé     : Lepirudin (ID: DB00001)
Score      : 1.0 ✅
Match type : exact

Query      : 'hirudin recombinant'
Trouvé     : Lepirudin (ID: DB00001)
Score      : 0.8972 ✅
Match type : semantic


In [ ]:
# ============================================================
# CELLULE 7 : Validation médicale complète
# (C'est ici que le RAG s'intègre au pipeline HTR)
# ============================================================

def validate_prescription_rag(extracted_text: str):
    """
    Prend le texte brut extrait par TrOCR + corrigé par BioMistral
    et valide chaque médicament via RAG DrugBank.

    Retourne un rapport de validation structuré.
    """
    import re

    # --- Étape 1 : Extraction des tokens candidats médicaments ---
    # (version simple — sera améliorée avec NER médical)
    tokens = re.findall(r'\b[A-Za-zÀ-ÿ]{4,}\b', extracted_text)
    tokens = list(set(tokens))  # Déduplique

    validated   = []
    uncertain   = []
    not_found   = []

    for token in tokens:
        results = search_drug(token, top_k=1, score_threshold=0.4)

        if not results:
            not_found.append(token)
            continue

        best = results[0]

        # Seuil de confiance
        if best["score"] >= 0.80:
            status = "validé"
            validated.append({
                "token_extrait": token,
                "medicament":    best["name"],
                "drugbank_id":   best["drugbank_id"],
                "score":         best["score"],
                "dosages":       best["dosages"],
                "status":        status
            })
        elif best["score"] >= 0.55:
            status = "incertain"
            uncertain.append({
                "token_extrait": token,
                "candidat":      best["name"],
                "score":         best["score"],
                "status":        status
            })
        else:
            not_found.append(token)

    # --- Rapport final ---
    rapport = {
        "texte_original":  extracted_text,
        "validés":         validated,
        "incertains":      uncertain,
        "non_identifiés":  not_found,
        "score_global":    round(
            len(validated) / max(len(tokens), 1), 2
        )
    }

    return rapport


def display_rapport(rapport):
    """Affichage lisible du rapport de validation."""
    print("="*60)
    print("RAPPORT DE VALIDATION MÉDICALE RAG")
    print("="*60)
    print(f"\nTexte analysé : {rapport['texte_original']}")
    print(f"Score global  : {rapport['score_global']}")

    print(f"\n✅ MÉDICAMENTS VALIDÉS ({len(rapport['validés'])}) :")
    for m in rapport["validés"]:
        dosages_str = " | ".join(
            [f"{d['strength']} {d['form']}" for d in m["dosages"][:2]]
        ) if m["dosages"] else "dosage non spécifié"
        print(f"  - {m['token_extrait']} → {m['medicament']} "
              f"(score: {m['score']}) | Dosages: {dosages_str}")

    print(f"\n⚠️  INCERTAINS — validation humaine requise ({len(rapport['incertains'])}) :")
    for m in rapport["incertains"]:
        print(f"  - {m['token_extrait']} → candidat: {m['candidat']} "
              f"(score: {m['score']})")

    print(f"\n❌ NON IDENTIFIÉS ({len(rapport['non_identifiés'])}) :")
    for t in rapport["non_identifiés"]:
        print(f"  - {t}")
    print("="*60)


# --- Test complet sur une ordonnance simulée ---
texte_simule = "Lepirudin 50mg IV pour thrombocytopenie heparinique Refludan"
rapport = validate_prescription_rag(texte_simule)
display_rapport(rapport)

RAPPORT DE VALIDATION MÉDICALE RAG

Texte analysé : Lepirudin 50mg IV pour thrombocytopenie heparinique Refludan
Score global  : 1.0

✅ MÉDICAMENTS VALIDÉS (5) :
  - Refludan → Lepirudin (score: 1.0) | Dosages: 20 mg Injection, solution, concentrate | 50 mg Injection, solution, concentrate
  - Lepirudin → Lepirudin (score: 1.0) | Dosages: 20 mg Injection, solution, concentrate | 50 mg Injection, solution, concentrate
  - thrombocytopenie → Eltrombopag (score: 0.9052) | Dosages: 100 mg/1 Tablet, film coated | 12.5 mg/1 Tablet, film coated
  - heparinique → Heparin Disaccharide Iii-S (score: 0.8934) | Dosages: dosage non spécifié
  - pour → Anistreplase (score: 0.849) | Dosages: 30 unit Powder, for solution

⚠️  INCERTAINS — validation humaine requise (0) :

❌ NON IDENTIFIÉS (0) :


In [ ]:
# ============================================================
# CELLULE 8 : Sauvegarde de l'index Qdrant sur Drive
# (Important pour ne pas tout recalculer à chaque session Colab)
# ============================================================

import pickle
import os

SAVE_DIR = "/content/drive/MyDrive/OCR_Prescription/rag_index"
os.makedirs(SAVE_DIR, exist_ok=True)

# Sauvegarde des embeddings et documents (Qdrant mémoire ne persiste pas)
np.save(os.path.join(SAVE_DIR, "embeddings.npy"), embeddings)

with open(os.path.join(SAVE_DIR, "documents.pkl"), "wb") as f:
    pickle.dump(documents, f)

print(f"✅ Index sauvegardé dans {SAVE_DIR}")
print(f"   - embeddings.npy  : {embeddings.shape}")
print(f"   - documents.pkl   : {len(documents)} documents")


# --- Fonction de rechargement (pour les prochaines sessions) ---
def reload_rag_index(save_dir):
    """Recharge l'index RAG sans recalculer les embeddings."""
    print("Rechargement de l'index RAG...")

    embeddings_loaded = np.load(os.path.join(save_dir, "embeddings.npy"))

    with open(os.path.join(save_dir, "documents.pkl"), "rb") as f:
        documents_loaded = pickle.load(f)

    # Recréation de la collection Qdrant en mémoire
    client_new = QdrantClient(":memory:")
    client_new.recreate_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=embeddings_loaded.shape[1],
            distance=Distance.COSINE
        )
    )

    points = [
        PointStruct(
            id=i,
            vector=emb.tolist(),
            payload=doc["metadata"] | {"text": doc["text"]}
        )
        for i, (doc, emb) in enumerate(zip(documents_loaded, embeddings_loaded))
    ]

    client_new.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"✅ Index rechargé ! {len(documents_loaded)} médicaments disponibles.")
    return client_new, documents_loaded

# Usage au début d'une nouvelle session :
# client, documents = reload_rag_index(SAVE_DIR)

✅ Index sauvegardé dans /content/drive/MyDrive/OCR_Prescription/rag_index
   - embeddings.npy  : (10562, 768)
   - documents.pkl   : 10562 documents
